# Модуль-ноутбук: `config`

Єдине джерело правди — схема, leakage-allowlist, пороги, шляхи. Інші ноутбуки підтягують його через `%run 00_config.ipynb`.

> ▶️ Запускай усі ноутбуки **з кореня репозиторію** `shouldipost/`.

In [ ]:
"""Single source of truth for the data contract, paths, and leakage allowlist.

Importing this module must have NO side effects beyond defining constants.
Every other module imports column names / thresholds from here so the
data-prep / train / inference stages cannot drift apart.
"""

from pathlib import Path

# ----------------------------------------------------------------------------- paths
ROOT = Path.cwd()
if ROOT.name == 'notebooks':   # якщо запущено з теки notebooks/
    ROOT = ROOT.parent
DATA_RAW = ROOT / "data" / "raw" / "train.csv"
DATA_PROCESSED = ROOT / "data" / "processed"
MODELS_DIR = ROOT / "models"
REPORTS_DIR = ROOT / "reports"
PLOTS_DIR = REPORTS_DIR / "plots"

MODEL_PATH = MODELS_DIR / "model.joblib"
METADATA_PATH = MODELS_DIR / "metadata.json"
REFERENCE_PATH = MODELS_DIR / "reference.parquet"  # neighbours for "similar videos"
METRICS_PATH = REPORTS_DIR / "metrics.json"

# HuggingFace source (CC BY-NC 4.0 — research / non-commercial only)
HF_REPO_ID = "datahiveai/Tiktok-Videos"
HF_FILENAME = "train.csv"
HF_DIRECT_URL = (
    "https://huggingface.co/datasets/datahiveai/Tiktok-Videos/resolve/main/train.csv"
)

# ----------------------------------------------------------------------------- schema
# Available BEFORE posting — the ONLY columns features may ever touch.
PREPOST_COLS = ["description", "duration", "create_time", "author_unique_id", "author_id"]
# Identifiers — never features, never labels.
ID_COLS = ["url", "video_id"]
# Known only AFTER posting — LABEL-ONLY. features.py refuses to read these.
POSTHOC_COLS = [
    "play_count",
    "digg_count",
    "share_count",
    "comment_count",
    "collect_count",
    "repost_count",  # all-zero in this dataset; excluded from the engagement sum
]
# Engagement numerator for the label's engagement-rate (repost_count dropped: all zeros).
ENGAGEMENT_COLS = ["digg_count", "share_count", "comment_count", "collect_count"]

CREATOR_COL = "author_unique_id"
TIME_COL = "create_time"
DROP_COLS = ["repost_count", "location_created"]  # dropped per EDA (see PLAN.md §1)

# ----------------------------------------------------------------------------- behaviour
RANDOM_SEED = 42
TEST_FRACTION = 0.20          # newest 20% by create_time -> temporal holdout
MIN_PLAUSIBLE_EPOCH = 1_300_000_000  # ~2011-03; below this create_time is treated missing
MAX_PLAUSIBLE_EPOCH = 4_102_444_800  # ~2100; above this (incl. ms/µs epochs, inf) treated missing
DURATION_CLIP = (1.0, 180.0)  # winsorize seconds (p99≈151) -> bounds linear extrapolation
CHAR_LEN_CLIP = 300           # winsorize caption length so OOD captions can't saturate the model
WORD_LEN_CLIP = 60
ASSUME_TIMEZONE = "UTC"       # create_time epoch interpreted as UTC (documented assumption)

# Business decision: posting slots are scarce -> favour PRECISION of the "Post" class.
POST_PRECISION_TARGET = 0.60
# Minimum calibrated-confidence margin to ever commit: never say Post unless p>=0.55 and
# never say "Do not post" unless p<=0.45, so "Unsure" stays meaningful on weak signal.
MIN_DECISION_MARGIN = 0.05
# A threshold only "counts" toward the precision target if it also has enough support —
# otherwise chance tail-points on a handful of samples would satisfy any precision target.
THRESHOLD_MIN_RECALL = 0.05
THRESHOLD_MIN_SUPPORT = 20
# Last-resort band if even the percentile fallback degenerates.
FALLBACK_BAND = (0.45, 0.55)

# Which model to deploy. "auto" = pick the best-CALIBRATED candidate by train out-of-fold
# Brier score (honest probabilities are the product's core value). Can be forced to a
# specific name ("logreg_full" / "hgb"). The chosen name is recorded in metadata.json.
DEPLOY_MODEL = "auto"
DEPLOY_CANDIDATES = ["logreg_full", "hgb"]

In [ ]:
from types import SimpleNamespace
# пакуємо всі КОНСТАНТИ (імена у ВЕРХНЬОМУ регістрі) у простір імен `config`
config = SimpleNamespace(**{k: v for k, v in dict(globals()).items()
                            if k.isupper() and not k.startswith('_')})

### Перевірка / демо

In [ ]:
print('фічей у контракті визначає features; тут — константи')
print('seed:', config.RANDOM_SEED, '| test_fraction:', config.TEST_FRACTION)